In [1]:
# Load the packages
import requests
from bs4 import BeautifulSoup
import time
from urllib.request import urlretrieve
from itertools import chain
from collections import Counter
from urllib.error import URLError, HTTPError
#import fitz
import pymupdf
import pdftotext
import os
import json
import re

In [2]:
# Defining the url of the site
base_site = "https://www.ifrc.org/appeals?page=1"

# Making a get request
response = requests.get(base_site, headers={'User-Agent': 'Mozilla/5.0'})
response.status_code

# get the HTML from the webpage
html = response.content


# convert the HTML to a Beautiful Soup object
soup = BeautifulSoup(html, 'html.parser')

In [4]:
#n_pages_str = soup.find('a', {'title' : 'Go to last page'})["href"].strip("?page=")
n_pages = 499#int(n_pages_str)

In [5]:
n_pages

499

In [6]:
def get_reports_snippets(page_number):
    # Defining the url of the site
    base_site = "https://www.ifrc.org/appeals?page=" + str(page_number)

    # Making a get request
    response = requests.get(base_site, headers={'User-Agent': 'Mozilla/5.0'})
    response.status_code

    # get the HTML from the webpage
    html = response.content

    # convert the HTML to a Beautiful Soup object
    soup = BeautifulSoup(html, 'html.parser')

    # Find all div tags on the webpage containing the information we want to scrape
    trs = soup.find_all("tr")
    return(trs)

In [7]:
tr_1 = get_reports_snippets(page_number=1)

In [8]:
tr_1[2].find(class_="views-field views-field-appeal-name").get_text(strip=True)

IndexError: list index out of range

In [9]:
def get_reports_info(trs):
    ifrc_reports_info = []

    for tr in trs[1:]:

        report_info = {}
        # Name
        report_name_element = tr.find(class_="views-field views-field-appeal-name")
        report_name = report_name_element.get_text(strip=True) if report_name_element else None
        # Disaster type
        disaster_type_element = tr.find(class_="views-field views-field-disaster-type").contents[0]
        disaster_type = disaster_type_element.strip(' ') if disaster_type_element else None
        # Date
        time_element = tr.find('time', class_='datetime')
        # Extract the datetime attribute
        datetime_value = time_element['datetime'] if time_element else None
        # Extract the text content
        date_text = time_element.get_text(strip=True) if time_element else None
        # Link to download report
        link_element = tr.find(class_="views-field views-field-appeal-name").contents[0]
        # Extract the href attribute
        link = link_element['href'] if link_element else None
        # Location
        location = tr.find(class_="views-field views-field-organisational-structure").contents[0].strip(' ')
        # Appeal code
        appeal_code = tr.find(class_="views-field views-field-appeal-code").contents[0].strip(' ')
        # Appeal type
        appeal_type = tr.find(class_="views-field views-field-appeal-type").contents[0].strip(' ')

        report_info["reportName"] = report_name
        report_info["disasterType"] = disaster_type
        report_info["date"] = date_text
        report_info["reportLink"] = link
        report_info["location"] = location
        report_info["appealCode"] = appeal_code
        report_info["appealType"] = appeal_type

        ifrc_reports_info.append(report_info)
    return(ifrc_reports_info)

In [14]:
all_ifrc_reports_info = []
for page_number in range(1, n_pages+1):
    print(page_number)
    trs = get_reports_snippets(page_number)
    time.sleep(5)
    print("Sleeping for 5 seconds")
    ifrc_reports_info = get_reports_info(trs)
    all_ifrc_reports_info.append(ifrc_reports_info)

1
Sleeping for 5 seconds
2
Sleeping for 5 seconds
3
Sleeping for 5 seconds
4
Sleeping for 5 seconds
5
Sleeping for 5 seconds
6


KeyboardInterrupt: 

In [18]:
def check_hazard_type_keyword(text):
    text = text.lower()
    hazards = []
    if re.search(r"drought.*|dry spell.*", text, re.IGNORECASE):
        hazards.append('Drought')
    if re.search(r"flood.*|inundation.*", text, re.IGNORECASE):
        hazards.append('Flood')
    if re.search(r"storm.*|superstorm.*|windstorm.*|snowstorm.*|snowfal.*|blizzard.*|derecho.*|winterstorm.*|hail.*|extra tropical storm.*|thunderstorm.*|storm surge.*|typhoon.*", text, re.IGNORECASE):
        hazards.append('Storm')
    if re.search(r"typhoon.*", text, re.IGNORECASE):
        hazards.append('Typhoon')
    if re.search(r"tornado.*", text, re.IGNORECASE):
        hazards.append('Tornado')
    if re.search(r"hurricane.*", text, re.IGNORECASE):
        hazards.append('Hurricane')
    if re.search(r"heat wave.*|heatwave.*|heat episode.*|((heat|hot) spell).*|heat stress.*", text, re.IGNORECASE):
        hazards.append('Heatwave')
    if re.search(r"cold wave.*|coldwave.*|severe winter conditions.*|cold spell.*", text, re.IGNORECASE):
        hazards.append('Coldwave')
    if re.search(r"land slide.*|landslide.*|rockfall.*|mudslide.*|mass movement.*", text, re.IGNORECASE):
        hazards.append('Mass movement')
    if re.search(r"earthquake.*", text, re.IGNORECASE):
        hazards.append('Earthquake')
    if re.search(r"cyclone.*|tropical cyclone.*", text, re.IGNORECASE):
        hazards.append('Cyclone')
    if re.search(r"volcan.*", text, re.IGNORECASE):
        hazards.append('Volcano')
    if re.search(r"tidal wav.*", text, re.IGNORECASE):
        hazards.append('Tidal Wave')
    if re.search(r"fire.*|forestfire.*|wildfire.*|landfire.*|bushfire.*|forest fire.*|wild fire.*|land fire.*|bush fire.*", text, re.IGNORECASE):
        hazards.append('Wildfire')
    if re.search(r"humanitarian crisis.*", text, re.IGNORECASE):
        hazards.append('Humanitarian Crisis')
    if re.search(r"food.*|hung.*", text, re.IGNORECASE):
        hazards.append('Food Insecurity')
    if re.search(r"population movement.*|displac.*|pop. movement.*", text, re.IGNORECASE):
        hazards.append('Population Movement')
    if re.search(r"annual appeal*", text, re.IGNORECASE):
        hazards.append('Annual Appeal')
    if re.search(r"unified plan*", text, re.IGNORECASE):
        hazards.append('Unified Plan')
    if len(hazards) == 0:
        return 'Other'
    else:
        return ', '.join(hazards)

In [11]:
all_ifrc_reports_info_unnested = [d for sublist in all_ifrc_reports_info for d in sublist]

NameError: name 'all_ifrc_reports_info' is not defined

In [4]:
file_path = './Data/all_ifrc_reports_info_unnested_processed.json'

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    all_ifrc_reports_info_unnested = json.load(json_file)

In [5]:
appeals_to_open = []
for appeal in all_ifrc_reports_info_unnested:
    if appeal['reportName'] == 'Colombia - Floods (MDRCO022)':
        appeals_to_open.append(appeal)

In [6]:
appeals_to_open

[{'reportName': 'Colombia - Floods (MDRCO022)',
  'disasterType': 'Flood',
  'date': '22/01/2024',
  'reportLink': 'https://adore.ifrc.org/Download.aspx?FileId=793548',
  'location': 'Colombia',
  'appealCode': 'MDRCO022',
  'appealType': 'DREF Operation Final Report',
  'pdfDownloaded': 1,
  'text': 'DREF APPLICATION\nColombia: Floods\nSource: www.radionacional.co/noticias-colombia/uribia-la-guajira-afectados-por-lluvias-temporada-in-\nvernal\nAppeal:\nMDRCO022\nDREF Allocated:\nCHF 499,988\nCrisis Category:\nYellow\nHazard:\nFlood\nGlide Number:\nPeople Aﬀected:\n501,281 people\nPeople Targeted:\n25,353 people\nEvent Onset:\nSudden\nOperation Start Date:\n2022-12-08\nOperation End Date:\n2023-03-31\nOperation Timeframe:\n3 months\nTargeted Areas:\nCauca, Cundinamarca, Huila, La Guajira, Valle del \nCauca\nPage 1 / 17\nDescription of the Event\nDepartments targeted by CRC to respond to ﬂoods emergency through this DREF Plan of Action. Source: CRC.\nWhat happened, where and when?\nOn 1

Now, use the links to download the reports

In [11]:
for appeal in all_ifrc_reports_info_unnested:
    if appeal['reportName'] == 'Colombia - Floods (MDRCO022)':
        filename = '/Volumes/Untitled/red-cross-reports/data/reports/' + appeal['appealCode'] + '.pdf' #/Users/taiscarvalho/Documents/projects/red-cross-reports/data/reports/' + appeal['appealCode'] + '.pdf'
        url = appeal['reportLink']

        # Check if the file already exists
        if os.path.exists(filename):
            print(f"File already exists: {filename}")
            appeal['pdfDownloaded'] = 1
        else:
            if url:
                try:
                    urlretrieve(url, filename)
                    print(f"Downloaded: {filename}")
                    appeal['pdfDownloaded'] = 1
                except HTTPError as e:
                    print(f"HTTP Error: {e.code} - {e.reason} for URL: {url}")
                    appeal['pdfDownloaded'] = 0
                except URLError as e:
                    print(f"URL Error: {e.reason} for URL: {url}")
                    appeal['pdfDownloaded'] = 0
                except Exception as e:
                    print(f"Unexpected error: {str(e)} for URL: {url}")
                    appeal['pdfDownloaded'] = 0
            else:
                print(f"No URL provided for appeal code: {appeal['appealCode']}")
                appeal['pdfDownloaded'] = 0

        # Extract text from the PDF
        try:
            #pdf_document = pymupdf.open(filename)
            #text = ""
            #for page_num in range(len(pdf_document)):
            #    page = pdf_document.load_page(page_num)
            #    text += page.get_text("text")
            #pdf_document.close()
            text = ""
            with open(filename, "rb") as f:
                pdf_document = pdftotext.PDF(f)
                for page in pdf_document:
                    text += page
                    appeal['text'] = text




            print(f"Text extracted for appeal code: {appeal['appealCode']}")
        except Exception as e:
            print(f"Error extracting text from {filename}: {str(e)}")

File already exists: /Volumes/Untitled/red-cross-reports/data/reports/MDRCO022.pdf
Text extracted for appeal code: MDRCO022


In [27]:
# Save to a JSON file
with open('/Users/lseverino/Documents/PhD/Projects/Como/como_project4/Data/all_ifrc_reports_info_unnested_v3.json', 'w') as json_file:
    json.dump(all_ifrc_reports_info_unnested, json_file, indent=4)

In [28]:
for element in all_ifrc_reports_info_unnested:
    if element['disasterType'] == '-':
        element['disasterTypeFlag'] = 1
        disaster_type = check_hazard_type_keyword(element['reportName'])
        element['disasterType'] = disaster_type

In [29]:
for element in all_ifrc_reports_info_unnested:
    if element['disasterType'] == 'Other':
        element['disasterTypeFlag'] = 1
        disaster_type = check_hazard_type_keyword(element['reportName'])
        element['disasterType'] = disaster_type

In [30]:
for element in all_ifrc_reports_info_unnested:
    if element['disasterType'] == 'Other':
        if 'text' in element:
            element['disasterTypeFlag'] = 2
            disaster_type = check_hazard_type_keyword(element['text'][:200])
            element['disasterType'] = disaster_type

In [31]:
hazard_events_dict = {'Flood, Pluvial/Flash Flood': 'Flood',
                      'Pluvial/Flash Flood': 'Flood',
                      'All other disaster and emergencies': 'Other disaster',
                      'Cold Wave': 'Coldwave',
                      'Drought, All other disaster and emergencies': 'Drought, Other disaster',
                      'Heat Wave': 'Heatwave',
                      'Heat Wave, Fire': 'Heatwave, Fire',
                      'Cold Wave, All other disaster and emergencies': 'Coldwave, Other disaster',
                      'Storm Surge': 'Storm',
                      'Landslide': 'Mass movement',
                      'Volcanic Eruption': 'Volcano',
                      'Landslide, Flood, All other disaster and emergencies': 'Landslide, Flood, Other disaster',
                      'Drought, Flood, All other disaster and emergencies': 'Drought, Flood, Other disaster',
                      'Drought, Flood, Population Movement, All other disaster and emergencies': 'Drought, Flood, Population Movement, Other disaster',
                      'Pluvial/Flash Flood, All other disaster and emergencies': 'Flood, Other disaster',
                      'Heat Wave, Fire': 'Heatwave, Fire',
                      'Cyclone, Storm Surge': 'Cyclone, Storm',
                      'Cold Wave, Earthquake, All other disaster and emergencies': 'Coldwave, Earthquake, Other disaster',
                      'Landslide, Cyclone, Pluvial/Flash Flood': 'Landslide, Cyclone, Flood',
                      'Cyclone, All other disaster and emergencies': 'Cyclone, Other disaster',
                      'Cold Wave, Flood': 'Coldwave, Flood',
                      'Famine / Food Insecurity': 'Food Insecurity',
                      'Landslide, Cyclone, Pluvial/Flash Flood': 'Landslide, Cyclone, Flood',
                      'Epidemic, Flood, Population Movement, All other disaster and emergencies': 'Epidemic, Flood, Population Movement, Other disaster',
                      'Cold Wave, Heat Wave, Drought, Epidemic, Insect Infestation, Earthquake, Landslide, Tsunami, Volcanic Eruption, Cyclone, Storm Surge, Flood, Pluvial/Flash Flood, Transport Accident, Nuclear Emergency, Chemical Emergency, Civil Unrest, Complex Emergency, Fire, Population Movement, All other disaster and emergencies': 'Coldwave, Heatwave, Drought, Epidemic, Insect Infestation, Earthquake, Landslide, Tsunami, Volcanic Eruption, Cyclone, Storm, Flood, Transport Accident, Nuclear Emergency, Chemical Emergency, Civil Unrest, Complex Emergency, Fire, Population Movement, Other disaster',
                      'Drought, Epidemic, Earthquake, Cyclone, Flood, All other disaster and emergencies': 'Drought, Epidemic, Earthquake, Cyclone, Flood, Other disaster',
                      'Drought, Population Movement, All other disaster and emergencies': 'Drought, Population Movement, Other disaster',
                      'Epidemic, Flood, Population Movement, All other disaster and emergencies': 'Epidemic, Flood, Population Movement, Other disaster',
                      'Drought, Epidemic, All other disaster and emergencies': 'Drought, Epidemic, Other disaster',
                      'Epidemic, Population Movement, All other disaster and emergencies': 'Epidemic, Population Movement, Other disaster',
                      'Drought, Epidemic, Flood, Population Movement, All other disaster and emergencies': 'Drought, Epidemic, Flood, Population Movement, Other disaster',
                      'Flood, Population Movement, All other disaster and emergencies': 'Flood, Population Movement, Other disaster'}

In [32]:
for element in all_ifrc_reports_info_unnested:
    disaster_type = element['disasterType']
    element['disasterTypeReclassified'] = hazard_events_dict.get(disaster_type, disaster_type)

In [33]:
sum = 0
list_disasters = []
list_locations = []

for element in all_ifrc_reports_info_unnested:
    sum = sum + element['pdfDownloaded']
    list_disasters.append(element['disasterTypeReclassified'])
    list_locations.append(element['location'])

In [34]:
Counter(list_disasters)

Counter({'Other': 800,
         'Flood': 760,
         'Earthquake': 349,
         'Population Movement': 291,
         'Cyclone': 233,
         'Epidemic': 214,
         'Other disaster': 176,
         'Annual appeal': 157,
         'Drought': 118,
         'Food Insecurity': 107,
         'Storm': 47,
         'Civil Unrest': 47,
         'Volcano': 38,
         'Complex Emergency': 33,
         'Fire': 26,
         'Humanitarian Crisis': 24,
         'Coldwave': 21,
         'Hurricane': 21,
         'Annual Appeal': 20,
         'Unified Plan': 13,
         'Wildfire': 12,
         'Mass movement': 11,
         'Flood, Mass movement': 10,
         'Cyclone, Flood': 9,
         'Flood, Storm': 9,
         'Heatwave': 5,
         'Drought, Other disaster': 5,
         'Transport Accident': 4,
         'Epidemic, All other disaster and emergencies': 4,
         'Population Movement, All other disaster and emergencies': 4,
         'Tidal Wave': 4,
         'Cholera': 3,
         'Eart

In [35]:
# Save to a JSON file
with open('/Users/lseverino/Documents/PhD/Projects/Como/como_project4/Data/all_ifrc_reports_info_unnested_processed_v3.json', 'w') as json_file:
    json.dump(all_ifrc_reports_info_unnested, json_file, indent=4)